# 01 — Data Acquisition: Kepler/TESS Light Curves with Ground Truth Labels

Downloads real stellar light curves from MAST via lightkurve and creates per-point binary labels from known exoplanet ephemerides (NASA Exoplanet Archive).

**Requirements:** Internet must be enabled in Kaggle settings.

**Output:** `data/labeled/` with metadata.csv + per-target CSV files (time, flux, flux_err, label)

In [ ]:
# Setup: install deps and import codebase
!pip install -q lightkurve astroquery PyWavelets matplotlib

import sys, os, glob

# Auto-detect codebase path — handles any Kaggle dataset structure
CODEBASE_PATH = None
for candidate in [
    '/kaggle/input/exopattern-codebase',                          # direct upload
    *glob.glob('/kaggle/input/exopattern-codebase/*/'),           # nested one level (e.g. ExopatternV3-copy/)
    *glob.glob('/kaggle/input/exopattern*/'),                     # alternate dataset names
    *glob.glob('/kaggle/input/exopattern*/*/'),                   # alternate names, nested
]:
    if os.path.isdir(os.path.join(candidate, 'backend')):
        CODEBASE_PATH = candidate.rstrip('/')
        break

if CODEBASE_PATH is None:
    raise FileNotFoundError(
        "Could not find 'backend/' directory in /kaggle/input/. "
        "Upload the project root (containing backend/ and experiments/) as a Kaggle dataset."
    )

sys.path.insert(0, CODEBASE_PATH)
print(f"Codebase found at: {CODEBASE_PATH}")
print(f"Contents: {os.listdir(CODEBASE_PATH)}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
from backend.data.acquisition import MastDataAcquisitor

acquisitor = MastDataAcquisitor(output_dir='/kaggle/working/data/labeled')

## 1. Query Confirmed Planets

Query the NASA Exoplanet Archive for confirmed transiting planets discovered by Kepler.

In [ ]:
# Query planet hosts
planet_targets = acquisitor.query_confirmed_planets(max_targets=150, mission='Kepler')
print(f"Found {len(planet_targets)} confirmed planet hosts")
planet_targets.head(10)

## 2. Build Full Dataset

Download light curves and create ground truth labels for all targets.

Using `max_quarters=5` to balance data volume vs download time (~3000-10000 points per target).

In [ ]:
# Full dataset: 150 planet hosts, 0 non-planet (non-planet query has issues)
# max_quarters=5 for reasonable download time
metadata = acquisitor.build_dataset(
    n_planet_hosts=150,
    n_non_planet=0,
    mission='Kepler',
    delay_between=1.0,
    max_quarters=5,
)

print(f"\nDownloaded {len(metadata)} targets")
metadata.head(10)

## 3. Dataset Summary

In [ ]:
import json

with open('/kaggle/working/data/labeled/dataset_summary.json') as f:
    summary = json.load(f)
print(json.dumps(summary, indent=2))

print(f"\nTotal points: {metadata['n_points'].sum():,}")
print(f"Mean anomaly fraction: {metadata['anomaly_fraction'].mean()*100:.1f}%")
print(f"Median anomaly fraction: {metadata['anomaly_fraction'].median()*100:.1f}%")

In [ ]:
# Distribution of anomaly fractions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(metadata['anomaly_fraction'] * 100, bins=30, edgecolor='black')
axes[0].set_xlabel('Transit Fraction (%)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Transit Fractions')

axes[1].hist(metadata['n_points'], bins=30, edgecolor='black')
axes[1].set_xlabel('Number of Points')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Light Curve Lengths')

plt.tight_layout()
plt.savefig('/kaggle/working/data/labeled/dataset_distributions.pdf', bbox_inches='tight')
plt.show()

## 4. Visual Validation

Plot a few light curves with their transit labels to visually verify correctness.

In [ ]:
# Plot first 4 targets with transit labels
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=False)

for i, (_, row) in enumerate(metadata.head(4).iterrows()):
    lc = pd.read_csv(f"/kaggle/working/data/labeled/lightcurves/{row['filename']}")
    ax = axes[i]
    
    # Plot flux
    ax.plot(lc['time'], lc['flux'], '.', color='#333333', markersize=1, alpha=0.5, label='Flux')
    
    # Highlight transit points
    transit_mask = lc['label'] == 1
    if transit_mask.sum() > 0:
        ax.plot(lc['time'][transit_mask], lc['flux'][transit_mask], '.', 
                color='red', markersize=3, alpha=0.8, label=f'Transit ({transit_mask.sum()} pts)')
    
    ax.set_ylabel('Normalized Flux')
    ax.set_title(f"{row['target_id']} — {row.get('planet_name', 'N/A')} "
                 f"(P={row.get('period', 0):.2f}d, {row['anomaly_fraction']*100:.1f}% transit)")
    ax.legend(loc='lower right', fontsize=8)

axes[-1].set_xlabel('Time (BKJD)')
plt.tight_layout()
plt.savefig('/kaggle/working/data/labeled/validation_plots.pdf', bbox_inches='tight')
plt.show()

## 5. Phase-Folded Validation

Phase-fold a known planet to verify transit labels are centered on the dip.

In [ ]:
# Phase-fold the first target
row = metadata.iloc[0]
lc = pd.read_csv(f"/kaggle/working/data/labeled/lightcurves/{row['filename']}")

period = row['period']
epoch = row['epoch']

phase = ((lc['time'].values - epoch) % period) / period
# Shift so transit is at phase 0.5 for better visualization
phase = (phase + 0.5) % 1.0

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(phase, lc['flux'], '.', color='#333333', markersize=1, alpha=0.3)

transit_mask = lc['label'] == 1
ax.plot(phase[transit_mask], lc['flux'][transit_mask], '.', color='red', markersize=3, alpha=0.5)

ax.set_xlabel('Orbital Phase')
ax.set_ylabel('Normalized Flux')
ax.set_title(f"{row['target_id']} Phase-Folded (P={period:.4f} d)")
plt.tight_layout()
plt.savefig('/kaggle/working/data/labeled/phase_fold_validation.pdf', bbox_inches='tight')
plt.show()

print(f"Transit points should cluster around phase 0.5 (the dip).")
print(f"Transit fraction: {transit_mask.mean()*100:.1f}%")

## Done!

Dataset is saved to `/kaggle/working/data/labeled/`. Download the output to use in subsequent notebooks.

## 6. Package Output as ZIP

In [ ]:
import shutil

# Zip the entire labeled dataset for easy download/re-upload as Kaggle dataset
shutil.make_archive('/kaggle/working/exopattern-labeled-data', 'zip', '/kaggle/working/data')
zip_size = os.path.getsize('/kaggle/working/exopattern-labeled-data.zip') / (1024 * 1024)
print(f"Created exopattern-labeled-data.zip ({zip_size:.1f} MB)")
print("Download this ZIP and upload as a new Kaggle dataset for notebooks 02-05.")